# Scatter Results Visualization
This notebook creates an interactive scatterplot of experiment results versus a selected dataset criterion.

Select an experiment and a strategy from dropdown menus, then compare the results against one of the available dataset metrics:
- Percentage of numerical values
- Number of tables
- Number of columns

In [1]:
import json
import math
import os
from collections import defaultdict
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from IPython.display import clear_output, display

In [2]:
RESULTS_DIR = Path("results")
DATA_ROOT = Path("data")

available_experiments = sorted(
    [
        item.stem
        for item in RESULTS_DIR.glob("*.json")
        if item.is_file() and item.stem.endswith("_f1_score_null_strict")
    ]
)

available_experiment_options = [
    (item[:-len("_f1_score_null_strict")], item)
    for item in available_experiments
]

if not available_experiments:
    raise FileNotFoundError("No experiment JSON files found in the results/ directory with suffix _f1_score_null_strict.")

criterion_options = {
    "percentage_numerical": "Percentage of numerical values",
    "number_of_tables": "Number of tables",
    "number_of_columns": "Number of columns",
}

source_labels = {
    "bird": "BIRD",
    "spider": "Spider",
    "spider2": "Spider 2",
    "wikidb": "WikiDB",
    "other": "Other",
}

source_colors = {
    "bird": "#4477AA",
    "spider": "#66CCEE",
    "spider2": "#CCBB44",
    "wikidb": "#EE6677",
    "other": "#333333",
}


def consider_value(value):
    return value is not None and value != "None" and value != "nan"


def is_numerical(value):
    try:
        float(value)
        return True
    except Exception:
        return False


def is_valid_number(value):
    if value is None:
        return False
    if isinstance(value, str) and value.lower() == "nan":
        return False
    try:
        number = float(value)
    except Exception:
        return False
    return not math.isnan(number)


def get_gold_standard_path(database_name: str) -> Path:
    return DATA_ROOT / database_name / "gold_standard_results.json"


def load_gold_standard_stats(database_name: str) -> dict[str, float]:
    gold_path = get_gold_standard_path(database_name)
    if not gold_path.exists():
        raise FileNotFoundError(f"Could not find gold standard file for {database_name}: {gold_path}")

    with open(gold_path, encoding="utf-8") as f:
        gold_standard = json.load(f)

    num_tables = len(gold_standard)
    total_values = 0
    numerical_values = 0
    total_columns = 0

    for table in gold_standard.values():
        if table:
            total_columns += len(table[0])
        for row in table:
            for value in row:
                if consider_value(value):
                    total_values += 1
                    if is_numerical(value):
                        numerical_values += 1

    percentage_numerical = numerical_values / total_values if total_values > 0 else 0.0
    return {
        "percentage_numerical": percentage_numerical,
        "number_of_tables": num_tables,
        "number_of_columns": total_columns,
    }


def resolve_source_name(database_name: str) -> str:
    if database_name.startswith("bird_"):
        return "bird"
    if database_name.startswith("spider2_"):
        return "spider2"
    if database_name.startswith("spider_"):
        return "spider"
    if database_name.startswith("wikidb_"):
        return "wikidb"
    return "other"


def extract_strategy_metrics(experiment_name: str, strategy_name: str) -> list[dict]:
    file_path = RESULTS_DIR / f"{experiment_name}.json"
    if not file_path.exists():
        raise FileNotFoundError(f"Experiment file not found: {file_path}")

    with open(file_path, encoding="utf-8") as f:
        raw_results = json.load(f)

    points = []
    for database_name, db_results in sorted(raw_results.items(), key=lambda x: x[0]):
        if strategy_name not in db_results:
            continue
        strategy_values = db_results[strategy_name]
        if not isinstance(strategy_values, dict):
            continue

        for parameter, raw_value in sorted(strategy_values.items(), key=lambda x: x[0]):
            if isinstance(raw_value, dict):
                numeric_values = [
                    float(v)
                    for v in raw_value.values()
                    if is_valid_number(v)
                ]
                if not numeric_values:
                    continue
                value = sum(numeric_values) / len(numeric_values)
            else:
                if not is_valid_number(raw_value):
                    continue
                value = float(raw_value)

            points.append(
                {
                    "database": database_name,
                    "parameter": parameter,
                    "result_value": value,
                    "source": resolve_source_name(database_name),
                }
            )
    return points


def get_experiment_strategies(experiment_name: str) -> list[str]:
    file_path = RESULTS_DIR / f"{experiment_name}.json"
    if not file_path.exists():
        return []

    with open(file_path, encoding="utf-8") as f:
        raw_results = json.load(f)

    strategy_names = set()
    for db_results in raw_results.values():

        if isinstance(db_results, dict):    
            strategy_names.update(k for k in db_results.keys() if k != "gold_standard")
            
    return sorted(strategy_names)

In [3]:
def plot_experiment_scatter(experiment_name: str, strategy_name: str, criterion_key: str):
    points = extract_strategy_metrics(experiment_name, strategy_name)
    if not points:
        print(
            "No data points found for the selected experiment and strategy."
        )
        return

    grouped = defaultdict(lambda: {"x": [], "y": [], "text": []})
    x_label = criterion_options[criterion_key]
    y_label = "F1-Score (strict incl. NULL-values)"

    for point in points:
        try:
            stats = load_gold_standard_stats(point["database"])
            x_value = stats[criterion_key]
        except FileNotFoundError:
            continue

        grouped[point["source"]]["x"].append(x_value)
        grouped[point["source"]]["y"].append(point["result_value"])
        grouped[point["source"]]["text"].append(
            f"Database: {point['database']}<br>Parameter: {point['parameter']}<br>F1-Score: {point['result_value']:.4f}"
        )

    fig = go.Figure()
    all_x = []
    all_y = []
    for source_name, data in grouped.items():
        if not data["x"]:
            continue
        fig.add_trace(
            go.Scatter(
                x=data["x"],
                y=data["y"],
                mode="markers",
                marker={"color": source_colors.get(source_name, "#333333"), "size": 10},
                name=source_labels.get(source_name, source_name),
                text=data["text"],
                hoverinfo="text",
            )
        )
        all_x.extend(data["x"])
        all_y.extend(data["y"])

    if len(all_x) >= 2:
        try:
            coefficients = np.polyfit(all_x, all_y, 1)
            poly = np.poly1d(coefficients)
            x_line = sorted(all_x)
            y_line = poly(x_line)
            fig.add_trace(
                go.Scatter(
                    x=x_line,
                    y=y_line,
                    mode="lines",
                    line={"color": "black", "dash": "dash"},
                    name="Regression",
                )
            )
        except Exception:
            pass

    fig.update_layout(
        title=f"{experiment_name} — {strategy_name} vs {criterion_options[criterion_key]}",
        xaxis_title=x_label,
        yaxis_title=y_label,
        legend_title="Source",
        height=600,
    )
    fig.update_yaxes(range=[-0.1, 1.1], autorange=False)
    fig.show()

In [5]:
# Create the interactive widgets
experiment_dropdown = widgets.Dropdown(
    options=available_experiment_options,
    value=available_experiment_options[0][1],
    description="Experiment:",
    layout=widgets.Layout(width="80%"),
)

strategy_dropdown = widgets.Dropdown(
    options=get_experiment_strategies(available_experiment_options[0][1]),
    description="Strategy:",
    layout=widgets.Layout(width="80%"),
)

criterion_dropdown = widgets.Dropdown(
    options=[(label, key) for key, label in criterion_options.items()],
    value="percentage_numerical",
    description="Criterion:",
    layout=widgets.Layout(width="80%"),
)

output = widgets.Output()


def update_strategy_options(change=None):
    strategies = get_experiment_strategies(experiment_dropdown.value)
    if not strategies:
        strategy_dropdown.options = ["<no available strategy>"]
        strategy_dropdown.value = "<no available strategy>"
    else:
        strategy_dropdown.options = strategies
        if strategy_dropdown.value not in strategies:
            strategy_dropdown.value = strategies[0]


def render_plot(change=None):
    with output:
        clear_output(wait=True)
        if not strategy_dropdown.value or "<no available" in str(strategy_dropdown.value):
            print("No strategy selected for this experiment.")
            return
        plot_experiment_scatter(
            experiment_dropdown.value,
            strategy_dropdown.value,
            criterion_dropdown.value,
        )


def on_experiment_change(change):
    if change["type"] == "change" and change["name"] == "value":
        update_strategy_options()
        render_plot()


def on_strategy_or_criterion_change(change):
    if change["type"] == "change" and change["name"] == "value":
        render_plot()

experiment_dropdown.observe(on_experiment_change)
strategy_dropdown.observe(on_strategy_or_criterion_change)
criterion_dropdown.observe(on_strategy_or_criterion_change)

# Display the controls and initial plot
control_box = widgets.VBox([experiment_dropdown, strategy_dropdown, criterion_dropdown])
display(control_box)
display(output)

update_strategy_options()
render_plot()

Output()